In [ ]:
import scanpy as sc
import numpy as np
import os

input_dir = "./data_output/cortex_h5ad"
files = [f for f in os.listdir(input_dir) if f.endswith(".h5ad")]

N_HVG = 2000
N_PCS = 10

for f in files:
    in_path = os.path.join(input_dir, f)
    out_path = os.path.join(input_dir, f.replace(".h5ad", f".hvg{N_HVG}.pca{N_PCS}.h5ad"))

    print("Processing:", f)
    adata = sc.read_h5ad(in_path)

    if adata.n_vars < N_HVG:
        print(f"  ⚠️ n_vars={adata.n_vars} < {N_HVG}, skipping")
        continue

    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

    sc.pp.highly_variable_genes(
        adata,
        n_top_genes=N_HVG,
        flavor="seurat",
        subset=True
    )

    sc.pp.scale(adata, max_value=10)

    sc.tl.pca(adata, n_comps=N_PCS, svd_solver="arpack")

    Xpca = np.asarray(adata.obsm["X_pca"])[:, :N_PCS].copy()
    adata.obsm["X_pca"] = Xpca
    adata.obsm["PCA"] = Xpca

    adata.write_h5ad(out_path)
    print("  saved:", out_path)
    print("  X_pca shape:", adata.obsm["X_pca"].shape)
    print()


In [ ]:
result_dir = "./data_output/mmd_result"

data = {}

for folder in sorted(os.listdir(result_dir)):

    folder_path = os.path.join(result_dir, folder)

    if not os.path.isdir(folder_path):
        continue

    npy_path = os.path.join(
        folder_path,
        "mmd_distances_group_mmd_celltype.npy"
    )

    if not os.path.exists(npy_path):
        print("Missing:", npy_path)
        continue

    arr = np.load(npy_path)

    print(folder, "shape:", arr.shape)

    mmd = arr[:, :, 0]

    mmd_flat = mmd.flatten()

    data[folder] = mmd_flat

df = pd.DataFrame(data)

out_path = "./data_output/mmd_result/mmd_summary.csv"

df.to_csv(out_path, index=False)

print("Saved:", out_path)


In [ ]:
data_test = np.load("data_output/mmd_result/F1_NSS_vs_B73_SS/mmd_distances_group_mmd_celltype.npy")


In [ ]:
data_test[:,:,0]
